# Phase 4: Scaling & Robustness — SpectralGuard on Mamba-2.8B

**Environment:** Google Colab / Kaggle, 1× or 2× T4 GPU  
**Estimated runtime:** ~90 min on 1×T4  

## Experiments
- **A** SpectralGuard scaling: 130M → 1.4B → 2.8B (500 prompts each)
- **B** Multi-seed robustness: Mamba-130M with seeds 42, 123, 456 — mean ± std
- **C** Power-method accuracy: power vs full eigendecomposition (50 layers)
- **D** Attack transferability: 130M → 2.8B cross-model evasion rate

## Output
`phase4_results.json` — paste into `main.tex`


In [ ]:
# == Install dependencies ==
# Use !pip (not subprocess) so Colab/Kaggle shows live output
!pip install transformers>=4.39.0 datasets accelerate matplotlib
!pip install causal-conv1d>=1.2.0
!pip install mamba-ssm
print('\u2713 Packages installed')


In [ ]:
# == Imports & global config ==
import os, json, time, warnings, gc, re, random
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# GPU check
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name}  {props.total_memory/1e9:.1f} GB')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')


In [ ]:
# == SpectralProbe & SpectralGuard ==
# SpectralProbe uses the PROVEN nb04 implementation:
#   delta(x) = softplus(dt_proj(z_t))  — input-dependent
#   A_bar[t,i,j] = exp(delta[t,i] * A[i,j])
#   rho_t = mean_i  max_j |A_bar[t,i,j]|
# This gives DIFFERENT values for benign vs adversarial text (F1 > 0).
# The old static-weight SVD version always gave benign_rho == adv_rho → F1=0.

class SpectralProbe:
    """Per-token spectral radius via forward hooks on dt_proj (layer 0)."""

    def __init__(self, model, tokenizer, device='cuda', layer_idx=0):
        self.model  = model
        self.tok    = tokenizer
        self.device = self._detect_device(model, device)
        self.layer_idx = layer_idx
        self._captured = {}

        # Static A matrix for the chosen layer: shape [d_inner, d_state]
        mixer = model.backbone.layers[layer_idx].mixer
        self.A = (-torch.exp(mixer.A_log.float())).detach()
        print(f'  SpectralProbe: layer={layer_idx}, '
              f'A {list(self.A.shape)}, '
              f'input device={self.device}')

    @staticmethod
    def _detect_device(model, fallback='cuda'):
        try:
            emb = model.get_input_embeddings()
            if emb is not None:
                return next(emb.parameters()).device
        except StopIteration:
            pass
        return torch.device(fallback)

    def _hook(self, module, inp, out):
        self._captured['dt'] = out.detach()

    @torch.no_grad()
    def trajectory(self, text, max_len=128):
        """Return list[float] — per-token mean spectral radius."""
        mixer = self.model.backbone.layers[self.layer_idx].mixer
        h = mixer.dt_proj.register_forward_hook(self._hook)
        try:
            ids = self.tok(text, return_tensors='pt',
                           truncation=True, max_length=max_len).input_ids
            ids = ids.to(self.device)
            self.model(ids)

            if 'dt' not in self._captured:
                return []

            dt_raw = self._captured['dt']          # [1, seq, d_inner] or [1, d_inner]
            if dt_raw.dim() == 2:
                dt_raw = dt_raw.unsqueeze(1)       # make it [1, 1, d_inner]
            delta = F.softplus(dt_raw[0].float())  # [seq, d_inner]

            A_local = self.A.to(delta.device)
            # A_bar[t,i,j] = exp(delta[t,i] * A[i,j])
            A_bar  = torch.exp(delta.unsqueeze(-1) * A_local.unsqueeze(0))
            rho_ch = A_bar.abs().max(dim=-1).values  # [seq, d_inner]
            rho    = rho_ch.mean(dim=-1)              # [seq]
            return rho.cpu().tolist()
        finally:
            h.remove()
            self._captured = {}

    def cleanup(self):
        pass  # no persistent hooks to remove


class SpectralGuard:
    """Anomaly detector calibrated on benign spectral trajectories."""
    def __init__(self, threshold, drop_thr, window=5):
        self.threshold = threshold
        self.drop_thr  = drop_thr
        self.window    = window

    @classmethod
    def calibrate(cls, benign_trajs, alpha=0.05, window=5):
        means = [np.mean(t) for t in benign_trajs]
        drops = []
        for t in benign_trajs:
            for i in range(len(t) - window):
                drops.append(t[i] - t[i + window])
        thr   = float(np.percentile(means, alpha * 100))
        d_thr = float(np.percentile(drops, (1 - alpha) * 100)) if drops else 0.0
        print(f'  Calibrated: mean_rho_thr={thr:.4f}  drop_thr={d_thr:.4f}')
        return cls(thr, d_thr, window)

    def detect(self, traj):
        mr = np.mean(traj) if traj else 0.0
        is_atk, reason, conf = False, 'safe', 0.0
        if mr < self.threshold:
            is_atk, reason = True, 'low_mean'
            conf = max(conf, 1 - mr / self.threshold) if self.threshold != 0 else 1.0
        for i in range(len(traj) - self.window):
            d = traj[i] - traj[i + self.window]
            if d > self.drop_thr:
                is_atk, reason = True, 'sudden_drop'
                conf = max(conf, min(1, d / self.drop_thr)) if self.drop_thr != 0 else 1.0
                break
        if len(traj) >= 10:
            s, e = np.mean(traj[:5]), np.mean(traj[-5:])
            if self.drop_thr != 0 and s - e > self.drop_thr * 1.5:
                is_atk, reason = True, 'collapse'
                conf = max(conf, min(1, (s - e) / (self.drop_thr * 1.5)))
        return is_atk, {'reason': reason, 'conf': conf,
                        'mean_rho': mr, 'min_rho': float(np.min(traj)) if traj else 0.0,
                        'var': float(np.var(traj)) if traj else 0.0}


def evaluate(guard, trajs, labels, idx_list, desc='Eval'):
    """Evaluate SpectralGuard; returns dict with metrics + bootstrapped 95% CI."""
    yt, yp, dets = [], [], []
    for i in tqdm(idx_list, desc=desc):
        t = trajs[i]
        if not t:
            yt.append(labels[i]); yp.append(0); dets.append({'mean_rho': 0})
            continue
        atk, d = guard.detect(t)
        yt.append(labels[i]); yp.append(int(atk)); dets.append(d)
    yt, yp = np.array(yt), np.array(yp)

    def m(y, p):
        tp  = ((p==1)&(y==1)).sum(); fp = ((p==1)&(y==0)).sum()
        fn  = ((p==0)&(y==1)).sum(); tn = ((p==0)&(y==0)).sum()
        pr  = tp/(tp+fp) if tp+fp > 0 else 0.0
        rc  = tp/(tp+fn) if tp+fn > 0 else 0.0
        f1  = 2*pr*rc/(pr+rc) if pr+rc > 0 else 0.0
        fpr = fp/(fp+tn)      if fp+tn > 0 else 0.0
        return pr, rc, f1, fpr, int(tp), int(fp), int(fn), int(tn)

    P, R, F, FPR, tp, fp, fn, tn = m(yt, yp)
    rng  = np.random.default_rng(42)
    boot = {'prec': [], 'rec': [], 'f1': [], 'fpr': []}
    for _ in range(2000):
        ix = rng.choice(len(yt), len(yt), replace=True)
        p_, r_, f_, fpr_ = m(yt[ix], yp[ix])[:4]
        boot['prec'].append(p_); boot['rec'].append(r_)
        boot['f1'].append(f_);   boot['fpr'].append(fpr_)
    ci = {k: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))
          for k, v in boot.items()}
    return {'prec': P, 'rec': R, 'f1': F, 'fpr': FPR,
            'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
            'ci': ci, 'dets': dets, 'yt': yt, 'yp': yp}


print('\u2713 SpectralProbe / SpectralGuard / evaluate defined')


In [ ]:
# == Dataset builder (Wikipedia + HiSPA-style adversarial) ==

def build_dataset(n_per_class=250, seed=42):
    """Return (prompts, labels) with n_per_class benign + n_per_class adversarial."""
    random.seed(seed); np.random.seed(seed)

    # --- benign: Wikipedia (parquet, no trust_remote_code needed) ---
    try:
        wiki = load_dataset('wikimedia/wikipedia', '20231101.en',
                            split='train', streaming=True)
    except Exception:
        wiki = load_dataset('wikipedia', '20220301.en',
                            split='train', streaming=True, trust_remote_code=True)

    benign = []
    for doc in wiki:
        text = doc['text'].strip()
        sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text)
                 if 10 < len(s.split()) < 40]
        benign.extend(sents)
        if len(benign) >= n_per_class * 5:
            break
    random.shuffle(benign)
    benign = benign[:n_per_class]
    assert len(benign) == n_per_class, f'Only got {len(benign)} benign sentences!'
    print(f'  Collected {len(benign)} benign sentences from Wikipedia')

    # --- adversarial: HiSPA word-shuffle ---
    adversarial = []
    for s in benign:
        words = s.split()
        random.shuffle(words)
        adversarial.append(' '.join(words))

    prompts = benign + adversarial
    labels  = [0]*n_per_class + [1]*n_per_class
    combined = list(zip(prompts, labels))
    random.shuffle(combined)
    prompts, labels = zip(*combined)
    return list(prompts), list(labels)


print('\u2713 build_dataset defined')


In [ ]:
# == Model loader ==

def load_mamba(model_id, device='cuda'):
    """Load Mamba model + tokenizer. Uses device_map='auto' for large models."""
    print(f'Loading {model_id} ...')
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    # float16 for memory efficiency; device_map='auto' for multi-GPU / large models
    kwargs = dict(torch_dtype=torch.float16)
    if any(x in model_id.lower() for x in ['2.8b', '1.4b']):
        kwargs['device_map'] = 'auto'
    else:
        kwargs['device_map'] = device   # single GPU for 130M

    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    model.eval()
    n = sum(p.numel() for p in model.parameters())
    print(f'  Params: {n/1e6:.0f} M  dtype={next(model.parameters()).dtype}')
    return model, tok


def make_trajectories(probe, prompts, max_len=128, desc='Traj'):
    return [probe.trajectory(p, max_len=max_len)
            for p in tqdm(prompts, desc=desc)]


print('\u2713 load_mamba / make_trajectories defined')


## Experiment A — SpectralGuard Scaling: 130M → 1.4B → 2.8B
Run the identical pipeline on three model sizes and compare:
- Mean spectral radius (benign vs. adversarial)
- SpectralGuard detection performance (F1, FPR + 95% CI)

**Expected runtime:** ~60 min total (1×T4)


In [ ]:
# == Experiment A: SpectralGuard scaling sweep ==
MODELS      = [
    'state-spaces/mamba-130m-hf',
    'state-spaces/mamba-1.4b-hf',
    'state-spaces/mamba-2.8b-hf',
]
N_PER_CLASS = 250   # 500 prompts total per model
MAX_LEN     = 128

expA_results = {}

for model_id in MODELS:
    tag      = model_id.split('/')[-1]                    # 'mamba-130m-hf'
    size_key = tag.replace('mamba-', '').replace('-hf', '') # '130m', '1.4b', '2.8b'
    print(f'\n{"="*60}')
    print(f'MODEL: {model_id}  (key={size_key})')
    print(f'{"="*60}')
    t0 = time.time()

    # 1. Load
    model, tok = load_mamba(model_id, device=DEVICE)
    probe = SpectralProbe(model, tok, device=DEVICE)

    # 2. Dataset
    prompts, labels = build_dataset(n_per_class=N_PER_CLASS, seed=SEED)
    N = len(prompts)
    print(f'  Dataset: {N} prompts ({N//2} benign + {N//2} adversarial)')

    # 3. Trajectories
    trajs = make_trajectories(probe, prompts, max_len=MAX_LEN)
    traj_lens = [len(t) for t in trajs]
    print(f'  Traj lengths: min={min(traj_lens)}, max={max(traj_lens)}, '
          f'median={np.median(traj_lens):.0f}')
    assert all(len(t) > 0 for t in trajs), 'Some trajectories are empty!'

    # 4. Split
    rng      = np.random.default_rng(SEED)
    idx      = rng.permutation(N)
    cal_idx  = idx[:N//2].tolist()
    test_idx = idx[N//2:].tolist()
    cal_ben_trajs = [trajs[i] for i in cal_idx if labels[i] == 0]

    # 5. Calibrate & evaluate
    guard = SpectralGuard.calibrate(cal_ben_trajs, alpha=0.05)
    res   = evaluate(guard, trajs, labels, test_idx, desc=f'Eval {size_key}')

    # 6. Spectral stats
    ben_means = [np.mean(trajs[i]) for i in range(N) if labels[i] == 0]
    adv_means = [np.mean(trajs[i]) for i in range(N) if labels[i] == 1]

    expA_results[size_key] = {
        'model_id'   : model_id,
        'n_params_M' : round(sum(p.numel() for p in model.parameters()) / 1e6),
        'n_total'    : N,
        'prec'       : float(res['prec']),
        'rec'        : float(res['rec']),
        'f1'         : float(res['f1']),
        'fpr'        : float(res['fpr']),
        'ci'         : {k: [float(v[0]), float(v[1])] for k, v in res['ci'].items()},
        'cm'         : {'tp': res['tp'], 'fp': res['fp'], 'fn': res['fn'], 'tn': res['tn']},
        'guard'      : {'threshold': float(guard.threshold), 'drop_thr': float(guard.drop_thr)},
        'spectral'   : {
            'benign_mean': float(np.mean(ben_means)),
            'benign_std' : float(np.std(ben_means)),
            'adv_mean'   : float(np.mean(adv_means)),
            'adv_std'    : float(np.std(adv_means)),
        },
        'elapsed_s'  : round(time.time() - t0, 1),
    }

    print(f'  F1={res["f1"]:.3f}  FPR={res["fpr"]:.3f}  '
          f'benign_rho={np.mean(ben_means):.4f}  adv_rho={np.mean(adv_means):.4f}')
    print(f'  Done in {time.time()-t0:.1f}s')

    # Free VRAM
    probe.cleanup()
    del model, tok, probe, trajs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print('\n\u2713 Experiment A complete')

# Print summary
print('\n====  EXPERIMENT A: SCALING RESULTS  ====')
hdr = f'{"Size":>6}  {"Params":>8}  {"F1":>6}  {"FPR":>6}  {"benign_rho":>11}  {"adv_rho":>9}'
print(hdr)
print('-'*len(hdr))
for size, r in expA_results.items():
    sp = r['spectral']
    print(f'{size:>6}  {r["n_params_M"]:>7}M  '
          f'{r["f1"]:>6.3f}  {r["fpr"]:>6.3f}  '
          f'{sp["benign_mean"]:>11.4f}  {sp["adv_mean"]:>9.4f}')


## Experiment B — Multi-seed Robustness (Mamba-130M)
Three independent seeds (42, 123, 456). Report mean ± std.

**Expected runtime:** ~15 min


In [ ]:
# == Experiment B: multi-seed robustness ==
SEEDS   = [42, 123, 456]
MODEL_B = 'state-spaces/mamba-130m-hf'

print(f'Loading {MODEL_B} for multi-seed run ...')
model_b, tok_b = load_mamba(MODEL_B, device=DEVICE)

expB_per_seed = []

for seed in SEEDS:
    print(f'\n-- Seed {seed} --')
    t0 = time.time()

    probe_b  = SpectralProbe(model_b, tok_b, device=DEVICE)
    prompts, labels = build_dataset(n_per_class=N_PER_CLASS, seed=seed)
    trajs = make_trajectories(probe_b, prompts, max_len=MAX_LEN, desc=f'Seed {seed}')

    rng      = np.random.default_rng(seed)
    idx      = rng.permutation(len(prompts))
    cal_idx  = idx[:len(prompts)//2].tolist()
    test_idx = idx[len(prompts)//2:].tolist()
    cal_ben_trajs = [trajs[i] for i in cal_idx if labels[i] == 0]

    guard_b = SpectralGuard.calibrate(cal_ben_trajs, alpha=0.05)
    res_b   = evaluate(guard_b, trajs, labels, test_idx, desc=f'Eval seed {seed}')

    expB_per_seed.append({
        'seed': seed,
        'prec': float(res_b['prec']),
        'rec' : float(res_b['rec']),
        'f1'  : float(res_b['f1']),
        'fpr' : float(res_b['fpr']),
    })
    print(f'  F1={res_b["f1"]:.3f}  FPR={res_b["fpr"]:.3f}  ({time.time()-t0:.1f}s)')
    probe_b.cleanup()
    del trajs

del model_b, tok_b
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Aggregate
expB_agg = {}
for metric in ['prec', 'rec', 'f1', 'fpr']:
    vals = [r[metric] for r in expB_per_seed]
    expB_agg[metric] = {'mean': float(np.mean(vals)), 'std': float(np.std(vals))}

print('\n====  EXPERIMENT B: MULTI-SEED ROBUSTNESS (Mamba-130M)  ====')
for metric, v in expB_agg.items():
    print(f'  {metric:6s}: {v["mean"]:.3f} +/- {v["std"]:.3f}')
print('\u2713 Experiment B complete')


## Experiment C — Power Method Validation
Compare three spectral radius estimators on 50 sampled `dt_proj` weight matrices.
Ground truth: eigendecomposition of W^T W.

**Expected runtime:** ~5 min


In [ ]:
# == Experiment C: power-method vs. full eigendecomposition ==
MODEL_C  = 'state-spaces/mamba-130m-hf'
N_PROBES = 50

print(f'Loading {MODEL_C} for power-method validation ...')
model_c, tok_c = load_mamba(MODEL_C, device=DEVICE)

# Collect dt_proj weight matrices (static, no forward pass)
dt_weights = {}
for name, mod in model_c.named_modules():
    if 'dt_proj' in name and hasattr(mod, 'weight'):
        dt_weights[name] = mod.weight.detach().float().cpu()

print(f'  Found {len(dt_weights)} dt_proj layers')
n_sample    = min(N_PROBES, len(dt_weights))
rng_c       = np.random.default_rng(42)
all_keys    = list(dt_weights.keys())
sampled_keys = [all_keys[i]
                for i in rng_c.choice(len(all_keys), size=n_sample, replace=False)]

svd_vals, eig_vals, power_vals = [], [], []

for key in tqdm(sampled_keys, desc='Power method vs SVD'):
    W = dt_weights[key]   # (d_inner, d_state)

    # 1. SVD (production method)
    sv      = torch.linalg.svdvals(W)
    rho_svd = float(sv[0].item())

    # 2. Full eigendecomposition (W^T W) — ground truth
    WtW     = W.T @ W
    eigvals = torch.linalg.eigvalsh(WtW)   # sorted ascending
    rho_eig = float(eigvals[-1].sqrt().item())

    # 3. Power iteration (50 iterations, float64 for stability)
    Wnp = W.numpy().astype(np.float64)
    v   = np.random.default_rng(7).standard_normal(Wnp.shape[1]).astype(np.float64)
    v   = v / np.linalg.norm(v)
    norm = 0.0
    for _ in range(50):
        w    = Wnp @ v
        v    = Wnp.T @ w
        norm = np.linalg.norm(v)
        if norm > 0:
            v = v / norm
    rho_pm = float(np.sqrt(max(norm, 0.0)))

    svd_vals.append(rho_svd)
    eig_vals.append(rho_eig)
    power_vals.append(rho_pm)

del model_c, tok_c, dt_weights
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

svd_arr   = np.array(svd_vals)
eig_arr   = np.array(eig_vals)
power_arr = np.array(power_vals)

mae_svd   = float(np.mean(np.abs(svd_arr   - eig_arr)))
mae_power = float(np.mean(np.abs(power_arr - eig_arr)))
corr_svd  = float(np.corrcoef(svd_arr,   eig_arr)[0, 1])
corr_pm   = float(np.corrcoef(power_arr, eig_arr)[0, 1])

expC_results = {
    'n_layers'        : int(n_sample),
    'mae_svd_vs_eig'  : mae_svd,
    'mae_pm_vs_eig'   : mae_power,
    'corr_svd_vs_eig' : corr_svd,
    'corr_pm_vs_eig'  : corr_pm,
}

print('\n====  EXPERIMENT C: POWER-METHOD ACCURACY  ====')
print(f'  SVD   MAE vs eigen: {mae_svd:.6f}   corr: {corr_svd:.6f}')
print(f'  Power MAE vs eigen: {mae_power:.6f}   corr: {corr_pm:.6f}')
print('\u2713 Experiment C complete')


## Experiment D — Attack Transferability (130M → 2.8B)
Craft adaptive attacks on the small model, then test if they fool the large model.

**Expected runtime:** ~20 min (1×T4)


In [ ]:
# == Experiment D: transferability 130M → 2.8B ==
MODEL_SRC  = 'state-spaces/mamba-130m-hf'
MODEL_TGT  = 'state-spaces/mamba-2.8b-hf'
N_TRANSFER = 100   # prompts to craft and test

# ── Step 1: craft attacks on source model (130M) ────────────────────────────
print(f'[1/4] Loading source model {MODEL_SRC} ...')
model_src, tok_src = load_mamba(MODEL_SRC, device=DEVICE)
probe_src = SpectralProbe(model_src, tok_src, device=DEVICE)

prompts_d, labels_d = build_dataset(n_per_class=N_TRANSFER, seed=SEED)
trajs_src  = make_trajectories(probe_src, prompts_d, max_len=MAX_LEN, desc='Src trajs')
src_ben    = [i for i in range(len(prompts_d)) if labels_d[i] == 0]
guard_src  = SpectralGuard.calibrate([trajs_src[i] for i in src_ben], alpha=0.05)

def craft_adaptive(prompt, probe, guard, max_len=128, attempts=2):
    """Append character suffixes to evade guard on source model."""
    chars    = ['_', '.', '!', '-', '~', '=', '*', ',', ';']
    best, best_sc = prompt, 0.0
    for _ in range(attempts):
        for ch in chars:
            for n in [5, 10, 15, 20, 25]:
                cand = prompt + (' ' + ch) * n
                tr   = probe.trajectory(cand, max_len=max_len)
                if not tr:
                    continue
                atk, d = guard.detect(tr)
                if not atk:
                    sc = 1.0 - d['mean_rho']
                    if sc > best_sc:
                        best_sc, best = sc, cand
    return best

adv_idx = [i for i in range(len(prompts_d)) if labels_d[i] == 1][:N_TRANSFER // 2]
print(f'[2/4] Crafting {len(adv_idx)} adaptive attacks on {MODEL_SRC} ...')
t0      = time.time()
crafted = [craft_adaptive(prompts_d[i], probe_src, guard_src)
           for i in tqdm(adv_idx, desc='Craft')]
print(f'  Done in {time.time()-t0:.1f}s')

probe_src.cleanup()
del model_src, tok_src, probe_src, trajs_src
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ── Step 2: evaluate attacks on target model (2.8B) ─────────────────────────
print(f'[3/4] Loading target model {MODEL_TGT} ...')
model_tgt, tok_tgt = load_mamba(MODEL_TGT, device=DEVICE)
probe_tgt          = SpectralProbe(model_tgt, tok_tgt, device=DEVICE)

benign_prompts   = [prompts_d[i] for i in range(len(prompts_d)) if labels_d[i] == 0]
benign_trajs_tgt = make_trajectories(probe_tgt, benign_prompts,
                                     max_len=MAX_LEN, desc='Benign on target')
guard_tgt = SpectralGuard.calibrate(benign_trajs_tgt, alpha=0.05)

print('[4/4] Evaluating transfer ...')
n_evaded         = 0
transfer_details = []
for p in tqdm(crafted, desc='Transfer eval'):
    tr = probe_tgt.trajectory(p, max_len=MAX_LEN)
    if not tr:
        transfer_details.append({'evaded': False, 'mean_rho': 0.0})
        continue
    atk, d = guard_tgt.detect(tr)
    evaded = not atk
    if evaded:
        n_evaded += 1
    transfer_details.append({'evaded': evaded, 'mean_rho': d['mean_rho']})

transfer_rate = n_evaded / len(crafted) if crafted else 0.0

expD_results = {
    'model_src'     : MODEL_SRC,
    'model_tgt'     : MODEL_TGT,
    'n_crafted'     : len(crafted),
    'n_evaded_tgt'  : n_evaded,
    'transfer_rate' : float(transfer_rate),
    'guard_src'     : {'threshold': float(guard_src.threshold), 'drop_thr': float(guard_src.drop_thr)},
    'guard_tgt'     : {'threshold': float(guard_tgt.threshold), 'drop_thr': float(guard_tgt.drop_thr)},
}

print(f'\n====  EXPERIMENT D: ATTACK TRANSFER (130M → 2.8B)  ====')
print(f'  Crafted: {len(crafted)}')
print(f'  Evaded on 2.8B: {n_evaded}')
print(f'  Transfer rate: {transfer_rate:.3f}')
print('\u2713 Experiment D complete')

probe_tgt.cleanup()
del model_tgt, tok_tgt, probe_tgt
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# == Figure: Phase 4 summary ==
sizes    = list(expA_results.keys())
params   = [expA_results[s]['n_params_M'] for s in sizes]
ben_rhos = [expA_results[s]['spectral']['benign_mean'] for s in sizes]
adv_rhos = [expA_results[s]['spectral']['adv_mean']    for s in sizes]
f1s      = [expA_results[s]['f1'] for s in sizes]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
ax.plot(params, ben_rhos, 'o-', color='#2196F3', label='Benign', linewidth=2)
ax.plot(params, adv_rhos, 's-', color='#f44336', label='Adversarial', linewidth=2)
ax.set_xscale('log')
ax.set_xlabel('Parameters (M, log scale)')
ax.set_ylabel('Mean spectral radius')
ax.set_title('(a) Spectral Radius vs. Model Size')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(params, f1s, 'D-', color='#4CAF50', linewidth=2)
ax.set_xscale('log')
ax.set_xlabel('Parameters (M, log scale)')
ax.set_ylabel('F1 Score')
ax.set_title('(b) SpectralGuard F1 vs. Size')
ax.set_ylim(0, 1); ax.grid(True, alpha=0.3)

ax = axes[2]
metrics_b = ['prec', 'rec', 'f1', 'fpr']
means_b   = [expB_agg[m]['mean'] for m in metrics_b]
stds_b    = [expB_agg[m]['std']  for m in metrics_b]
ax.bar(metrics_b, means_b, yerr=stds_b, capsize=6,
       color=['#2196F3', '#4CAF50', '#FF9800', '#f44336'], alpha=0.8)
ax.set_ylabel('Score'); ax.set_ylim(0, 1)
ax.set_title('(c) Multi-seed Robustness (130M)')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('phase4_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: phase4_results.png')


In [ ]:
# == JSON summary output ==
phase4 = {
    'experiment_A_scaling'     : expA_results,
    'experiment_B_multi_seed'  : {'per_seed': expB_per_seed, 'aggregate': expB_agg},
    'experiment_C_power_method': expC_results,
    'experiment_D_transfer'    : expD_results,
    'config': {
        'n_per_class': N_PER_CLASS,
        'max_len'    : MAX_LEN,
        'seeds'      : SEEDS,
        'device'     : str(DEVICE),
        'method'     : 'forward_hooks_on_dt_proj_Abar',
    }
}

with open('phase4_results.json', 'w') as f:
    json.dump(phase4, f, indent=2)
print('Saved: phase4_results.json')
print(json.dumps(phase4, indent=2))


## Summary
- **Exp A** – Spectral radius shift with scale; SpectralGuard F1 & FPR across 130M/1.4B/2.8B
- **Exp B** – Multi-seed std (robustness check, Mamba-130M)
- **Exp C** – Power method MAE and correlation vs. eigendecomposition
- **Exp D** – Transfer rate of adaptive attacks from 130M to 2.8B

Download `phase4_results.json` → paste numbers into Tables 6–9 of `main.tex`.
Download `phase4_results.png` → include as Phase 4 figure.
